## **Data Processing With No Limits**
### Note, run ThimkersRemoteWork.ipynb first before running this
### You must also use the exact same kernel to keep the variables

After getting the best hyperparameter combinations from the models with a 1000 iteration limit, we will now apply them to models with no iteration limit to see if it affects the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [2]:
%store -r
print("Variables restored successfully.")

Variables restored successfully.


In [3]:
print("Current Variables")
print(f"target                      : {target.shape}")
print(f"current_profession_encoded  : {current_profession_encoded.shape}")
print(f"age_group                   : {age_group.shape}")
print(f"education_level             : {education_level.shape}")
print(f"employment_status           : {employment_status.shape}")
print(f"dev_type_encoded            : {dev_type_encoded.shape}")
print(f"work_years (non-null)       : {work_years.notna().sum()}")
print(f"learn_years (non-null)      : {learn_years.notna().sum()}")
print(f"org_size_ordinal            : {org_size_ordinal.shape}")
print(f"work_tool_count             : {work_tool_count.shape}")
print(f"personal_tool_count         : {personal_tool_count.shape}")
print(f"geographic_regions_encoded  : {geographic_regions_encoded.shape}")
print(f"language_features           : {language_features.shape}")
print(f"database_features           : {database_features.shape}")
print(f"platform_features           : {platform_features.shape}")
print(f"webframe_features           : {webframe_features.shape}")
print(f"devenv_features             : {devenv_features.shape}")
print(f"collab_features             : {collab_features.shape}")
print(f"aimodel_features            : {aimodel_features.shape}")
print(f"ai_industry_use             : {ai_industry_use.shape}")
print(f"ai_learn_how                : {ai_learn_how.shape}")
print(f"learncodeai_encoded         : {learncodeai_encoded.shape}")
print(f"aiselect_encoded            : {aiselect_encoded.shape}")
print(f"aiagents_encoded            : {aiagents_encoded.shape}")
print(f"aiagentchange_encoded       : {aiagentchange_encoded.shape}")
print(f"ai_technical_use            : {ai_technical_use.shape}")
print(f"ai_knowledge                : {ai_knowledge.shape}")
print(f"ai_orchestration            : {ai_orchestration.shape}")
print(f"ai_observe_secure           : {ai_observe_secure.shape}")
print(f"ai_external                 : {ai_external.shape}")

Current Variables
target                      : (49191,)
current_profession_encoded  : (49191, 4)
age_group                   : (49191, 6)
education_level             : (49191, 8)
employment_status           : (49191, 5)
dev_type_encoded            : (49191, 21)
work_years (non-null)       : 42893
learn_years (non-null)      : 43042
org_size_ordinal            : (49191,)
work_tool_count             : (49191,)
personal_tool_count         : (49191,)
geographic_regions_encoded  : (49191, 19)
language_features           : (49191, 42)
database_features           : (49191, 30)
platform_features           : (49191, 42)
webframe_features           : (49191, 28)
devenv_features             : (49191, 27)
collab_features             : (49191, 25)
aimodel_features            : (49191, 17)
ai_industry_use             : (49191, 10)
ai_learn_how                : (49191, 13)
learncodeai_encoded         : (49191, 2)
aiselect_encoded            : (49191, 4)
aiagents_encoded            : (49191, 4)
aiage

## **All Features and Train/Test Split**

In [4]:
X = pd.concat([
    current_profession_encoded,
    age_group,
    education_level,
    employment_status,
    dev_type_encoded,
    geographic_regions_encoded,
    pd.DataFrame({'org_size': org_size_ordinal}),
    pd.DataFrame({'work_exp': work_years}),
    pd.DataFrame({'years_code': learn_years}),
    pd.DataFrame({'work_tools': work_tool_count}),
    pd.DataFrame({'personal_tools': personal_tool_count}),
    language_features,
    database_features,
    platform_features,
    webframe_features,
    devenv_features,
    collab_features,
    aimodel_features,
    ai_industry_use,
    ai_learn_how,
    learncodeai_encoded,
    aiselect_encoded,
    aiagents_encoded,
    aiagentchange_encoded,
    ai_technical_use,
    ai_knowledge,
    ai_orchestration,
    ai_observe_secure,
    ai_external,
], axis=1)

y = target

# Remove NaN and set median data
X_clean = X.copy()
X_clean = X_clean.fillna(0)

# 
for col in ['work_exp', 'years_code', 'work_tools', 'personal_tools', 'org_size']:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"Full matrix: {X_clean.shape}")
print(f"Target: {y.shape}")
print(f"Target Ratio: {y.value_counts().to_dict()}")

Full matrix: (49191, 395)
Target: (49191,)
Target Ratio: {0: 34016, 1: 15175}


In [5]:
# Three-way split: 60% train, 20% validation, 20% test
# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: separate train and validation from remaining 80%
# 0.25 of 80% = 20% of total for validation, leaving 60% for training
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Scale ONLY continuous features!!!
continuous_features = ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Fit scaler on training data only, transform all three sets
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_val_scaled[continuous_features] = scaler.transform(X_val[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Convert to numpy arrays
X_train_scaled = X_train_scaled.values
X_val_scaled = X_val_scaled.values
X_test_scaled = X_test_scaled.values

# Oversampling with SMOTE
# Performed Worse
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print(f"Train size: {X_train_scaled.shape}")
print(f"Validation size: {X_val_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"\nTrain class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation class balance: {pd.Series(y_val).value_counts().to_dict()}")
print(f"Test class balance: {pd.Series(y_test).value_counts().to_dict()}")
print(f"\nScaled features: {continuous_features}")
print(f"One-hot encoded features remain as 0/1 (not scaled)")

Train size: (29514, 395)
Validation size: (9838, 395)
Test size: (9839, 395)

Train class balance: {0: 20409, 1: 9105}
Validation class balance: {0: 6803, 1: 3035}
Test class balance: {0: 6804, 1: 3035}

Scaled features: ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']
One-hot encoded features remain as 0/1 (not scaled)


## **Logistic Regression with no Iteration Limit**

In [6]:
best_combo = ('l1', 'liblinear', {})
best_C = 0.1

label = f"{best_combo[0]}/{best_combo[1]}"
print(f"Fitting best combo: {label} with C={best_C}")
    
model = LogisticRegression(penalty='l1', solver='liblinear', C=best_C, random_state=42, **{})
model.fit(X_train_scaled, y_train)

Fitting best combo: l1/liblinear with C=0.1


c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` p

In [7]:
lr_predictions = model.predict(X_test_scaled)

cm_lr = confusion_matrix(y_test, lr_predictions)
acc_lr = accuracy_score(y_test, lr_predictions)
report_lr = classification_report(y_test, lr_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Metric', 'Non-Remote', 'Remote']),
    cells=dict(values=[
        ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        [f"{acc_lr:.4f}", f"{report_lr['Non-Remote']['precision']:.4f}", f"{report_lr['Non-Remote']['recall']:.4f}", f"{report_lr['Non-Remote']['f1-score']:.4f}"],
        [f"{acc_lr:.4f}", f"{report_lr['Remote']['precision']:.4f}", f"{report_lr['Remote']['recall']:.4f}", f"{report_lr['Remote']['f1-score']:.4f}"]
    ])
))
fig.update_layout(
    title=f'Logistic Regression ({label}, C={best_C}) - Classification Report',
    template='plotly_white',
    height=450
)
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=cm_lr,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_lr), str(fp_lr)], [str(fn_lr), str(tp_lr)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Logistic Regression ({label}, C={best_C}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()

In [8]:
# Getting top variables with .coef
feature_names = X_clean.columns.tolist()
coefs = model.coef_[0] 

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs
})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).head(20)
coef_df = coef_df.sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'royalblue' for c in coef_df['Coefficient']]

fig = go.Figure(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title=f'Logistic Regression ({label}, C={best_C}) - Top 20 Feature Coefficients',
    xaxis_title='Coefficient Value',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by absolute coefficient:")
print(f"{'Rank':<6} {'Feature':<40} {'Coefficient':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(coef_df.sort_values('Abs', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Coefficient']:>12.4f}")

Top 20 features by absolute coefficient:
Rank   Feature                                   Coefficient
------------------------------------------------------------
1      learncodeai_no                                 1.6667
2      learncodeai_yes                                1.6228
3      employment_employed                            1.3648
4      employment_independent                         0.9165
5      devtype_mobile developer                       0.7969
6      devtype_frontend developer                     0.7963
7      devtype_backend developer                      0.6495
8      region_south_america                           0.6465
9      employment_student                             0.6336
10     region_eastern_europe                          0.6177
11     region_central_america                         0.6063
12     region_eastern_asia                           -0.5968
13     devtype_qa tester                              0.5826
14     devtype_game developer               

## **SVM with no Iteration Limit**

In [9]:
SVM_C = 1
kernel = 'rbf'

train_errors, val_errors = [], []
print(f"Testing SVM with kernel: {kernel}")

svm = SVC(kernel=kernel, C=SVM_C, random_state=42)
svm.fit(X_train_scaled, y_train)
train_errors.append(1 - svm.score(X_train_scaled, y_train))
val_errors.append(1 - svm.score(X_val_scaled, y_val))
print(f"  C=1  Train Error: {train_errors[-1]:.4f}  Validation Error: {val_errors[-1]:.4f}")

svm_predictions = svm.predict(X_test_scaled)
cm_svm = confusion_matrix(y_test, svm_predictions)
acc_svm = accuracy_score(y_test, svm_predictions)
report_svm = classification_report(y_test, svm_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Metric', 'Non-Remote', 'Remote']),
    cells=dict(values=[
        ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        [f"{acc_svm:.4f}", f"{report_svm['Non-Remote']['precision']:.4f}", f"{report_svm['Non-Remote']['recall']:.4f}", f"{report_svm['Non-Remote']['f1-score']:.4f}"],
        [f"{acc_svm:.4f}", f"{report_svm['Remote']['precision']:.4f}", f"{report_svm['Remote']['recall']:.4f}", f"{report_svm['Remote']['f1-score']:.4f}"]
    ])
))
fig.update_layout(
    title=f'SVM (Kernel={kernel}, C={SVM_C}) - Classification Report',
    template='plotly_white',
    height=450
)
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=cm_svm,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_svm,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title='SVM Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    template='plotly_white'
)
fig.show()

Testing SVM with kernel: rbf
  C=1  Train Error: 0.1559  Validation Error: 0.2611


## **Neural Network with no Iteration Limit**

In [10]:
architecture = (256, 128, 64)
activation = 'tanh'
alpha = 0.01

mlp_best = MLPClassifier(
    hidden_layer_sizes=architecture,
    activation=activation,
    solver='adam',
    alpha=alpha,
    random_state=42
)

mlp_best.fit(X_train_scaled, y_train)
mlp_predictions = mlp_best.predict(X_test_scaled)

cm_mlp = confusion_matrix(y_test, mlp_predictions)
acc_mlp = accuracy_score(y_test, mlp_predictions)
report_mlp = classification_report(y_test, mlp_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_mlp, fp_mlp, fn_mlp, tp_mlp = cm_mlp.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Metric', 'Non-Remote', 'Remote'], fill_color='paleturquoise', align='left'),
    cells=dict(values=[
        ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        [f"{acc_mlp:.4f}", f"{report_mlp['Non-Remote']['precision']:.4f}", f"{report_mlp['Non-Remote']['recall']:.4f}", f"{report_mlp['Non-Remote']['f1-score']:.4f}"],
        [f"{acc_mlp:.4f}", f"{report_mlp['Remote']['precision']:.4f}", f"{report_mlp['Remote']['recall']:.4f}", f"{report_mlp['Remote']['f1-score']:.4f}"],
    ], fill_color='lavender', align='left')
))
fig.update_layout(
    title=f'Neural Network - Classification Report',
    template='plotly_white',
    height=400
)
fig.show()

fig = go.Figure(data=go.Heatmap(
    z=cm_mlp,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_mlp), str(fp_mlp)], [str(fn_mlp), str(tp_mlp)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Neural Network - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()

It seems that the models just predict Non-Remote because it is the majority class, which makes it very imprecise. Let us take a look at the precision of the models after SMOTE.

In [11]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print(f"Train size: {X_train_scaled.shape}")
print(f"Validation size: {X_val_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"\nTrain class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation class balance: {pd.Series(y_val).value_counts().to_dict()}")
print(f"Test class balance: {pd.Series(y_test).value_counts().to_dict()}")
print(f"\nScaled features: {continuous_features}")
print(f"One-hot encoded features remain as 0/1 (not scaled)")

Train size: (40818, 395)
Validation size: (9838, 395)
Test size: (9839, 395)

Train class balance: {0: 20409, 1: 20409}
Validation class balance: {0: 6803, 1: 3035}
Test class balance: {0: 6804, 1: 3035}

Scaled features: ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']
One-hot encoded features remain as 0/1 (not scaled)


In [13]:
model = LogisticRegression(penalty='l1', solver='liblinear', C=best_C, random_state=42, **{})
model.fit(X_train_scaled, y_train)
lr_predictions = model.predict(X_test_scaled)

cm_lr = confusion_matrix(y_test, lr_predictions)
acc_lr = accuracy_score(y_test, lr_predictions)
report_lr = classification_report(y_test, lr_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

svm = SVC(kernel=kernel, C=SVM_C, random_state=42)
svm.fit(X_train_scaled, y_train)
train_errors.append(1 - svm.score(X_train_scaled, y_train))
val_errors.append(1 - svm.score(X_val_scaled, y_val))
svm_predictions = svm.predict(X_test_scaled)

cm_svm = confusion_matrix(y_test, svm_predictions)
acc_svm = accuracy_score(y_test, svm_predictions)
report_svm = classification_report(y_test, svm_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()

mlp_best = MLPClassifier(
    hidden_layer_sizes=architecture,
    activation=activation,
    solver='adam',
    alpha=alpha,
    random_state=42
)
mlp_best.fit(X_train_scaled, y_train)
mlp_predictions = mlp_best.predict(X_test_scaled)

cm_mlp = confusion_matrix(y_test, mlp_predictions)
acc_mlp = accuracy_score(y_test, mlp_predictions)
report_mlp = classification_report(y_test, mlp_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_mlp, fp_mlp, fn_mlp, tp_mlp = cm_mlp.ravel()

fig = go.Figure(data=go.Table(
    header=dict(values=['Model', 'Non-Remote', 'Remote'], fill_color='paleturquoise', align='left'),
    cells=dict(values=[['Logistic Regression', acc_lr, acc_lr],
                       ['SVM', acc_svm, acc_svm],
                       ['MLP', acc_mlp, acc_mlp]]))
)
fig.update_layout(
    title=f'Model Comparison - Accuracy',
    template='plotly_white',
    height=300
)
fig.show()

fig = go.Figure(data=go.Table(
    header=dict(values=['Model', 'Non-Remote', 'Remote'], fill_color='paleturquoise', align='left'),
    cells=dict(values=[['Logistic Regression', f"{report_lr['Non-Remote']['precision']:.4f}", f"{report_lr['Remote']['precision']:.4f}"],
                       ['SVM', f"{report_svm['Non-Remote']['precision']:.4f}", f"{report_svm['Remote']['precision']:.4f}"],
                       ['MLP', f"{report_mlp['Non-Remote']['precision']:.4f}", f"{report_mlp['Remote']['precision']:.4f}"]])
))
fig.update_layout(
    title=f'Model Comparison - Precision',
    template='plotly_white',
    height=300
)
fig.show()

c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning:

'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.

c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning:

Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.



SMOTE really does make the models perform worse.